Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

In [33]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [34]:
##create again
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model ="groq:qwen/qwen3-32b",
    checkpointer = InMemorySaver(),
   middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)


In [35]:
config={"configurable":{"thread_id":"test-1"}}

In [37]:
## messsages
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"messages:{len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let me see. The user is asking "What is 15-7?" So, I need to calculate 15 minus 7. Let me start by recalling basic subtraction. 15 minus 7. Hmm, 7 plus 8 is 15, so 15 minus 7 should be 8. Let me verify that. If I take 7 away from 15, I\'m left with 8. Yes, that makes sense.\n\nWait, is there any chance the user is using a different base system here? Like in base 10 or another base. But since the previous questions were all in base 10, and the user didn\'t specify, I should stick with base 10. So 15 in base 10 minus 7 in base 10 is definitely 8. \n\nAlternatively, maybe the user is testing if I can handle negative numbers or something else. But the question is straightforward. There\'s no indication of a trick. The answer should be 8. Let me double-check by counting up from 7 to 15. 7 plus 8 is 15. Yep, so 15 minus 7 is 8. I think that\'s solid. No other factors to consider her

## token Size

In [38]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

## Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [68]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_mail_to(emailId:str)->str:
    """This is a mock function to read the email by Email Id"""
    return f"Email Content by Id : {emailId}"

def send_email_to(recipient:str,subject:str,body:str)->str:
    """This is a mock function to send the email"""
    f"Email has sent to {recipient} with subject:{subject} and body:{body}"
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    tools=[read_mail_to, send_email_to],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_to":{
                    "allowed_decisions":["approve","reject","edit"]
                },
                "read_mail_to":False
            }

        )
    ]
)
config={"configurable":{"thread_id":"1"}}
response = agent.invoke({"messages":HumanMessage(content="send email to test@gmail.com with subject 'testing mail ' and body :'Hello how are youu?'")}, config=config)

In [54]:
response

{'messages': [HumanMessage(content="send email to test@gmail.com with subject 'testing mail ' and body :'Hello how are youu?'", additional_kwargs={}, response_metadata={}, id='3a9a3ab0-a9a4-48a5-b901-c45864a5e348'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to test@gmail.com with a specific subject and body. Let me check the tools available. There's the send_email_to function. It requires recipient, subject, and body. The parameters are all there: recipient is test@gmail.com, subject is 'testing mail', and the body is 'Hello how are youu?'. I need to make sure all required fields are included. Yep, all three are required. So I should call send_email_to with these parameters. No issues here. The read_mail_to function isn't needed for this request. Alright, time to structure the tool call correctly in JSON within the XML tags.\n", 'tool_calls': [{'id': 'j8gdk7gbn', 'function': {'arguments': '{"body":"Hello how are youu?","recip

In [56]:
response["__interrupt__"]

[Interrupt(value={'action_requests': [{'name': 'send_email_to', 'args': {'body': 'Hello how are youu?', 'recipient': 'test@gmail.com', 'subject': 'testing mail '}, 'description': "Tool execution requires approval\n\nTool: send_email_to\nArgs: {'body': 'Hello how are youu?', 'recipient': 'test@gmail.com', 'subject': 'testing mail '}"}], 'review_configs': [{'action_name': 'send_email_to', 'allowed_decisions': ['approve', 'reject', 'editing']}]}, id='2457b9cfe33d93135e719bcf2195d52f')]

In [58]:
from langgraph.types import Command
if "__interrupt__" in response:
    print("paused.....")
    response = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "approve"}
                ]
            }
        ),
        config
    )
print(f"✅ Result: {response['messages'][-1].content}")

paused.....
✅ Result: The email has been successfully sent to **test@gmail.com** with the subject **"testing mail "** and the body **"Hello how are youu?"**. Let me know if you need any further assistance!


## rejected

In [63]:
# Step 2: Reject
if "__interrupt__" in response:
    print("⏸️ Paused! Approving...")
    
    response = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {response['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email could not be sent at this time. Please check your email settings or try again later. If you'd like to attempt sending it again, let me know!


## Editing

In [69]:
# Step 2: Edit and approve
if "__interrupt__" in response:
    print("⏸️ Paused! Editing...")
    
    response = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {response['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: 


In [70]:
response

{'messages': [HumanMessage(content="send email to test@gmail.com with subject 'testing mail ' and body :'Hello how are youu?'", additional_kwargs={}, response_metadata={}, id='9bb6b62c-13c5-42cc-9949-2ac09daca26d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let me break down what the user is asking. They want to send an email to test@gmail.com with a specific subject and body. The tools provided include a send_email_to function. I need to check the parameters required for that function. The required fields are recipient, subject, and body. The user provided all three: recipient is test@gmail.com, subject is 'testing mail', and the body is 'Hello how are youu?'. I should make sure the parameters match the function's requirements. There's no mention of needing to read an email, so the read_mail_to function isn't needed here. I'll structure the tool call with the send_email_to function and the provided details.\n", 'tool_calls': [{'id': 'ctxxne6j5', 'function'